# 09. Comparing External Frameworks

## Learning Objectives
- Understand the deeper differences between Deep Agents, LangGraph, and LangChain
- Compare them with OpenCode and the Claude Agent SDK
- Analyze differences in architecture, flexibility, and ecosystem
- Learn which framework to recommend for different use cases
- Understand migration considerations


In [ ]:
# Environment setup
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set!"
print("Environment setup complete")


In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")

print(f"Model configured: {model.model_name}")


---
## 1. Comparison Overview

When choosing an AI agent framework, you need to consider several factors, including **model support**, **architecture**, **ecosystem**, and **license**.

In this notebook, you compare three major options:

- **LangChain Deep Agents** — a model-agnostic agent harness
- **OpenCode** — a coding agent environment centered on the terminal / desktop / IDE
- **Claude Agent SDK** — Anthropic's SDK for Claude-based agents


---
## 2. Deep Agents vs OpenCode vs Claude Agent SDK

### Basic Comparison

| Feature | LangChain Deep Agents | OpenCode | Claude Agent SDK |
|------|----------------------|----------|------------------|
| **Model support** | Model-agnostic (Anthropic, OpenAI, 100+ providers) | 75+ providers, including local models through Ollama | Claude-only |
| **License** | MIT | MIT | MIT (SDK), proprietary (Claude Code) |
| **SDK** | Python, TypeScript + CLI | Terminal, desktop, IDE integration | Python, TypeScript |
| **Sandboxing** | Can be integrated as a tool | Not supported | Not supported |
| **State management** | Supports time travel | Not supported | Supports time travel |
| **Observability** | Native LangSmith support | None | None |


In [ ]:
# Print a framework comparison table
frameworks = {
    "LangChain Deep Agents": {
        "model_support": "100+ providers (model-agnostic)",
        "license": "MIT",
        "sdk": "Python, TypeScript, CLI",
        "sandbox": "Integrated support",
        "time_travel": "Supported",
    },
    "OpenCode": {
        "model_support": "75+ providers (including local models)",
        "license": "MIT",
        "sdk": "Terminal, desktop, IDE",
        "sandbox": "Not supported",
        "time_travel": "Not supported",
    },
    "Claude Agent SDK": {
        "model_support": "Claude only",
        "license": "MIT (SDK)",
        "sdk": "Python, TypeScript",
        "sandbox": "Not supported",
        "time_travel": "Supported",
    },
}

print("=== Framework Comparison ===")
for name, features in frameworks.items():
    print(f"\n[{name}]")
    for key, value in features.items():
        print(f"  {key}: {value}")


---
## 3. Comparing Core Capabilities

### Shared Capabilities
All three frameworks support the following categories:
- file operations (read, write, edit)
- shell command execution
- search features (`grep`, `glob`)
- planning support (task lists)
- Human-in-the-Loop (with different permission models)

### Differentiating Capabilities

| Capability | Deep Agents | OpenCode | Claude Agent SDK |
|------|------------|----------|------------------|
| **Core tools** | Files, shell, search, planning | Files, shell, search, planning | Files, shell, search, planning |
| **Sandbox integration** | Can be integrated as a tool | No | No |
| **Pluggable backends** | Storage + filesystem backends | No | No |
| **Virtual filesystem** | Yes, through pluggable backends | No | No |
| **Native tracing** | LangSmith | No | No |


---
## 4. Architecture Comparison

### LangChain Deep Agents
- **Pluggable storage backends** — state, filesystem, and store layers can be configured independently
- **Virtual filesystem** — can switch between local, in-memory, or sandboxed backends
- **LangGraph-based runtime** — supports complex workflows through graph execution
- **Middleware system** — fine-grained control over agent behavior

### OpenCode
- **Terminal-native** — lightweight and fast to start
- **75+ model providers** — includes local models through Ollama
- **LSP integration** — optimized for code editing workflows

### Claude Agent SDK
- **Claude-optimized** — tailored for Claude model capabilities
- **Time travel** — supports branching and state exploration
- **Concise API** — useful for fast prototyping


---
## 5. Recommendations by Use Case

| Use Case | Recommended Framework | Why |
|----------|--------------|------|
| Production agent app | **Deep Agents** | Pluggable backends, observability, sandbox support |
| Multi-model agent | **Deep Agents** | 100+ provider support |
| Terminal coding assistant | **OpenCode** | Lightweight startup, strong local-model story |
| Claude-only app | **Claude Agent SDK** | Optimized for Claude, simple API |
| Rapid prototyping | **Claude Agent SDK** | Minimal API, fast setup |
| Complex multi-agent system | **Deep Agents** | Subagents and strong context management |
| Local model usage | **OpenCode** | Native Ollama support |


In [ ]:
# Helper to recommend a framework by use case
def recommend_framework(use_case: str) -> str:
    """Recommend a framework for a given use case."""
    recommendations = {
        "production": ("Deep Agents", "pluggable backends, observability, sandbox support"),
        "multi-model": ("Deep Agents", "100+ provider support"),
        "terminal": ("OpenCode", "fast startup and strong support for local models"),
        "claude-only": ("Claude Agent SDK", "Claude optimization and a concise API"),
        "prototyping": ("Claude Agent SDK", "simple API and fast setup"),
        "multi-agent": ("Deep Agents", "subagents and context management"),
        "local-model": ("OpenCode", "native Ollama support"),
    }
    if use_case in recommendations:
        fw, reason = recommendations[use_case]
        return f"{fw} — {reason}"
    return "No recommendation found for that use case."

# Demo
for case in ["production", "terminal", "claude-only", "multi-agent"]:
    print(f"{case}: {recommend_framework(case)}")


---
## 6. Ecosystem Comparison

| Item | Deep Agents | OpenCode | Claude Agent SDK |
|------|------------|----------|------------------|
| **Community** | LangChain ecosystem (large) | GitHub community | Anthropic community |
| **Documentation** | Official docs + LangSmith integration | GitHub README | Anthropic official docs |
| **Integrations** | LangChain, LangGraph, LangSmith | LSP, terminal tooling | Claude API |
| **Package management** | pip / uv | go install / brew | pip / npm |
| **Editor integration** | ACP (Zed, JetBrains, VS Code, Neovim) | Own editor workflows | None |


---
## 7. Migration Considerations

Important questions when migrating between frameworks:

### Shared Considerations
1. **Model compatibility** — verify that the target framework supports the models you use
2. **Tool compatibility** — check how your custom tool interfaces need to be adapted
3. **State management** — plan how checkpoints and memory should be migrated
4. **Observability** — decide how tracing and logging should be replaced or preserved

### Advantages of Migrating to Deep Agents
- **LangChain tool reuse** — existing LangChain tools can often be reused directly
- **LangGraph compatibility** — integrates well with LangGraph-based workflows
- **Incremental migration** — supports gradual adoption rather than all-at-once rewrites

### Caveats
- Claude Agent SDK features that are Claude-specific may need alternative implementations
- Terminal UI logic from OpenCode may need to be separated from the core agent logic


---
## Summary

| Topic | Core Idea | Key API / Tool |
|------|----------|-------------|
| Three-way comparison | Deep Agents, OpenCode, Claude Agent SDK | model support, license, SDK |
| Core capabilities | Shared tools + differentiators | sandboxing, pluggable backends |
| Architecture | Pluggable vs terminal-native vs model-optimized | LangGraph, LSP, Claude API |
| Use-case guidance | Production, terminal, prototyping, etc. | `recommend_framework()` |
| Ecosystem | Community, docs, integrations, editor support | LangSmith, ACP |
| Migration | Model / tool / state compatibility | gradual adoption |

### Next Steps
→ **[10_sandboxes_and_acp.ipynb](./10_sandboxes_and_acp.ipynb)**


---
**References:**
- [Comparison with OpenCode and Claude Agent SDK](../docs/deepagents/04-comparison.md)
